In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
import sys

sys.path.append('/home/gracchus/code/UCAS-Deep-Learning-2026Spring/Lab4')
from NMTData import load_processed_data
from Evaluate import corpus_bleu


In [ ]:
# 设置路径，与前几个 Lab 的 notebook 保持类似写法
root = Path('/home/gracchus/code/UCAS-Deep-Learning-2026Spring')
lab4_dir = root / 'Lab4'
figure_dir = lab4_dir / 'Figure'
figure_dir.mkdir(exist_ok=True)

log_path = lab4_dir / 'logs/train_log.csv'
translation_path = lab4_dir / 'logs/test_translation.txt'
log_df = pd.read_csv(log_path)
log_df.head()


In [ ]:
# 读取预处理后的数据、词表以及测试集翻译结果
data, train_dataset, dev_dataset, src_vocab, tgt_vocab = load_processed_data()
predictions = [line.strip().split() for line in translation_path.open(encoding='utf-8')]

print(f'train samples: {len(train_dataset)}')
print(f'dev samples: {len(dev_dataset)}')
print(f'test samples: {len(data["test_sources"])}')
print(f'src vocab size: {len(src_vocab)}')
print(f'tgt vocab size: {len(tgt_vocab)}')


In [ ]:
# 汇总训练过程中的关键节点
best_dev = log_df.loc[log_df['dev_loss'].idxmin()]
final = log_df.iloc[-1]
bleu4 = corpus_bleu(predictions, data['references'], max_n=4)

print(f'best dev epoch: {int(best_dev["epoch"])}')
print(f'best dev loss: {best_dev["dev_loss"]:.4f}')
print(f'best dev ppl: {best_dev["dev_ppl"]:.2f}')
print(f'final epoch: {int(final["epoch"])}')
print(f'BLEU-4: {bleu4:.2f}')


In [ ]:
# 绘制训练损失和困惑度变化曲线
plt.style.use('default')

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(log_df['epoch'], log_df['train_loss'], label='Train Loss')
plt.plot(log_df['epoch'], log_df['dev_loss'], label='Dev Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training / Dev Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(log_df['epoch'], log_df['train_ppl'], label='Train PPL')
plt.plot(log_df['epoch'], log_df['dev_ppl'], label='Dev PPL')
plt.xlabel('Epoch')
plt.ylabel('Perplexity')
plt.title('Training / Dev Perplexity')
plt.legend()

plt.tight_layout()
plt.savefig(figure_dir / 'loss_ppl_curves.png', bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
# 绘制学习率和梯度范数变化曲线
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(log_df['epoch'], log_df['lr'])
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Learning Rate')

plt.subplot(1, 2, 2)
plt.plot(log_df['epoch'], log_df['grad_norm'])
plt.xlabel('Epoch')
plt.ylabel('Gradient Norm')
plt.title('Gradient Norm')

plt.tight_layout()
plt.savefig(figure_dir / 'lr_grad_curves.png', bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
# 比较参考译文和模型译文的长度分布
src_lens = np.array([len(x) for x in data['test_source_tokens']])
ref_lens = np.array([len(x) for x in data['references']])
pred_lens = np.array([len(x) for x in predictions])

print(f'avg source length: {src_lens.mean():.2f}')
print(f'avg reference length: {ref_lens.mean():.2f}')
print(f'avg prediction length: {pred_lens.mean():.2f}')

plt.figure(figsize=(8, 4))
bins = np.arange(0, 90, 5)
plt.hist(ref_lens, bins=bins, alpha=0.55, label='Reference')
plt.hist(pred_lens, bins=bins, alpha=0.55, label='Prediction')
plt.xlabel('Sentence Length')
plt.ylabel('Count')
plt.title('Reference / Prediction Length Distribution')
plt.legend()
plt.tight_layout()
plt.savefig(figure_dir / 'length_distribution.png', bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
# 选择几个测试句子，分别计算单句 BLEU-4
example_indices = [444, 917, 182, 733]
example_rows = []
for idx in example_indices:
    sent_bleu = corpus_bleu([predictions[idx]], [data['references'][idx]], max_n=4)
    example_rows.append({
        'index': idx,
        'source': ' '.join(data['test_source_tokens'][idx]),
        'prediction': ' '.join(predictions[idx]),
        'reference': ' '.join(data['references'][idx]),
        'sentence_bleu': sent_bleu,
    })

examples_df = pd.DataFrame(example_rows)
examples_df.to_csv(lab4_dir / 'logs/sentence_bleu_examples.csv', index=False)
examples_df[['index', 'sentence_bleu', 'source', 'prediction', 'reference']]


In [ ]:
# 绘制样例句子的单句 BLEU
plt.figure(figsize=(7, 4))
plt.bar([str(i) for i in examples_df['index']], examples_df['sentence_bleu'])
plt.xlabel('Test Sample Index')
plt.ylabel('Sentence BLEU-4')
plt.title('Sentence BLEU Examples')
plt.tight_layout()
plt.savefig(figure_dir / 'sentence_bleu_examples.png', bbox_inches='tight', dpi=300)
plt.show()
